# Technical Report Draft — Solar Filament Segmentation 2026

This notebook generates the technical report for the MAGFiLO 2026 Solar
Filament Segmentation Challenge from measured results rather than from prose
typed by hand.

Every number in the draft is read from `v2/results/summary.json`, which is
assembled from the experiment output files under `v2/runs/`. The draft
therefore cannot drift away from the measurements: if a result changes,
re-running this notebook changes the report.

**Why this matters for the rubric.** The qualitative half of the score is 30%
and is contingent on the code being available and the pipeline described
accurately. A report quoting numbers that no script produces is a liability.

## Structure

1. Environment setup
2. Imports
3. Report parameters
4. Load and inspect the results
5. Summary analysis
6. Report template
7. Render
8. Export to Markdown and HTML

## 1. Set up the notebook environment

Check the interpreter, set pandas display options so wide tables are readable,
and resolve the repository root. The notebook is expected to live in
`notebooks/`, so the root is one level up.

The output directory is created here rather than at export time so a failure
surfaces immediately instead of after the whole draft has been rendered.

In [1]:
import sys
from pathlib import Path

# Resolve the repo root from the notebook's own location, so the notebook
# works regardless of the directory Jupyter was launched from.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (ROOT / "v2" / "results").exists(), f"not a repo root: {ROOT}"

OUT_DIR = ROOT / "v2" / "results" / "report"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"python  {sys.version.split()[0]}")
print(f"root    {ROOT}")
print(f"output  {OUT_DIR}")

python  3.12.3
root    /media/yasir/HDD/drive/repos/competitions/solar
output  /media/yasir/HDD/drive/repos/competitions/solar/v2/results/report


## 2. Import required libraries

`jinja2` renders the report template and `markdown` converts the rendered
Markdown to HTML for the second export format.

In [2]:
import json
from datetime import date

import jinja2
import markdown
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

print(f"pandas   {pd.__version__}")
print(f"numpy    {np.__version__}")
print(f"jinja2   {jinja2.__version__}")
print(f"markdown {markdown.__version__}")

pandas   3.1.0
numpy    2.5.4
jinja2   3.1.6
markdown 3.11.3


## 3. Configure report parameters

One dictionary holds everything that identifies the report. Changing the
author or the source file here regenerates the whole draft, which is the point
of templating rather than writing prose.

In [3]:
CONFIG = {
    "title": "Solar Filament Segmentation with Panoptic Quality",
    "subtitle": "MAGFiLO 2026 Challenge — Technical Report",
    "author": "Yasir Latif",
    "report_date": date.today().isoformat(),
    "source": ROOT / "v2" / "results" / "summary.json",
    "out_md": OUT_DIR / "report_draft.md",
    "out_html": OUT_DIR / "report_draft.html",
}

for k, v in CONFIG.items():
    print(f"{k:12s} {v}")

title        Solar Filament Segmentation with Panoptic Quality
subtitle     MAGFiLO 2026 Challenge — Technical Report
author       Yasir Latif
report_date  2026-11-09
source       /media/yasir/HDD/drive/repos/competitions/solar/v2/results/summary.json
out_md       /media/yasir/HDD/drive/repos/competitions/solar/v2/results/report/report_draft.md
out_html     /media/yasir/HDD/drive/repos/competitions/solar/v2/results/report/report_draft.html


## 4. Load and inspect the data

The source is a JSON summary rather than a flat table, so it is loaded as a
dict and the per-phase results are flattened into a DataFrame for inspection.

There is a fallback: if the file is missing the notebook still runs, using a
small synthetic frame. That keeps the notebook executable on a fresh clone
before any experiments have been run, which matters because a reviewer may
open it before running anything.

In [4]:
if CONFIG["source"].exists():
    with open(CONFIG["source"]) as fh:
        SUMMARY = json.load(fh)
    print(f"loaded {CONFIG['source'].relative_to(ROOT)}")
else:
    print(f"WARNING: {CONFIG['source']} missing — using synthetic placeholder")
    SUMMARY = {
        "competition": {"name": "placeholder", "metric": "PQ"},
        "dataset": {"train_files": 0, "test_files": 0},
        "phases": [
            {"id": "phase1", "name": "placeholder", "change": "n/a",
             "pq": 0.0, "sq": 0.0, "rq": 0.0, "ci": None,
             "public_lb": None, "verdict": "unknown",
             "paired_delta": None, "paired_ci": None,
             "n_files": 0, "n_images": 0}
        ],
    }

# Flatten the phase list into a table. This is the frame the report is built
# from, so it is worth looking at directly before trusting the prose.
phases = pd.DataFrame(SUMMARY["phases"])
phases["ci"] = phases["ci"].apply(
    lambda c: f"[{c[0]:.4f}, {c[1]:.4f}]" if isinstance(c, list) else "—"
)
phases["paired_ci"] = phases["paired_ci"].apply(
    lambda c: f"[{c[0]:+.4f}, {c[1]:+.4f}]" if isinstance(c, list) else "—"
)

print(f"shape {phases.shape}")
print(f"columns {list(phases.columns)}")
print(f"nulls per column:\n{phases.isna().sum().to_string()}")

loaded v2/results/summary.json
shape (4, 16)
columns ['id', 'name', 'change', 'pq', 'sq', 'rq', 'ci', 'public_lb', 'verdict', 'n_files', 'n_images', 'paired_delta', 'paired_ci', 'note', 'blend_oof_pq', 'tta']
nulls per column:
id              0
name            0
change          0
pq              0
sq              0
rq              0
ci              0
public_lb       2
verdict         0
n_files         0
n_images        0
paired_delta    1
paired_ci       0
note            3
blend_oof_pq    3
tta             3


In [5]:
phases[["id", "name", "pq", "sq", "rq", "verdict", "n_files"]]

,id,name,pq,sq,rq,verdict,n_files
0,phase1,Segmentation baseline,0.4220,0.6765,0.6238,baseline,707
1,phase2,Candidate re-scoring,0.4386,0.6786,0.6464,accepted,707
2,phase3,GONG label transfer,0.4045,0.6667,0.6095,rejected,142
3,phase4,PVTv2-B2 ensemble diversity,0.4098,0.6719,0.6099,rejected,197


## 5. Run summary analysis

Compute the values the template needs. Keeping this separate from rendering
means the numbers can be checked before they are written into prose.

The most important derived quantity is the **oracle gap**: how much PQ is
available if every candidate were scored perfectly, and how much of that the
learned scorer actually captured. That framing is what makes the negative
results interpretable rather than just disappointing.

In [6]:
p1 = next(p for p in SUMMARY["phases"] if p["id"] == "phase1")
p2 = next(p for p in SUMMARY["phases"] if p["id"] == "phase2")
p3 = next(p for p in SUMMARY["phases"] if p["id"] == "phase3")
p4 = next(p for p in SUMMARY["phases"] if p["id"] == "phase4")

detail = SUMMARY.get("phase2_detail", {})
fp = SUMMARY.get("fp_anatomy", {})
p4d = SUMMARY.get("phase4_detail", {})
sub = SUMMARY.get("submission", {})
ds = SUMMARY["dataset"]

accepted = [p for p in SUMMARY["phases"] if p["verdict"] == "accepted"]
rejected = [p for p in SUMMARY["phases"] if p["verdict"] == "rejected"]

METRICS = {
    "n_phases": len(SUMMARY["phases"]),
    "n_accepted": len(accepted),
    "n_rejected": len(rejected),
    "best_pq": max(p["pq"] for p in SUMMARY["phases"]),
    "best_phase": max(SUMMARY["phases"], key=lambda p: p["pq"])["name"],
    "phase1_pq": p1["pq"],
    "phase2_pq": p2["pq"],
    "phase2_gain": p2["pq"] - p1["pq"],
    "phase2_paired": p2.get("paired_delta"),
    "phase2_paired_ci": p2.get("paired_ci"),
    "phase3_delta": p3.get("paired_delta"),
    "phase4_delta": p4.get("paired_delta"),
    "public_lb": sub.get("public_lb"),
    "local_public_gap": sub.get("local_public_gap"),
    "oracle_pq": detail.get("oracle_ceiling_pq"),
    "oracle_gap": detail.get("oracle_gap"),
    "captured_gap": detail.get("captured_gap"),
    "remaining_gap": detail.get("remaining_gap"),
    "capture_fraction": (
        detail.get("captured_gap") / detail.get("oracle_gap")
        if detail.get("oracle_gap") else None
    ),
    "n_candidates": detail.get("n_candidates"),
    "gbdt_corr": detail.get("gbdt_oof_corr"),
    "cnn_corr": detail.get("cnn_oof_corr"),
    "fragment_fraction": fp.get("fragment_fraction"),
    "fragment_oracle_gain": fp.get("fragment_oracle_gain"),
    "pvt_corr": p4d.get("mean_corr_conv_pvt"),
    "pvt_solo": p4d.get("pvtv2_solo_pq"),
    "conv_solo": p4d.get("convnext_solo_pq"),
    "train_files": ds.get("train_files"),
    "test_files": ds.get("test_files"),
    "n_annotations": ds.get("train_annotations"),
    "n_filaments_submitted": sub.get("n_filaments"),
    "n_empty_images": len(sub.get("empty_images", [])),
}

for k, v in METRICS.items():
    if isinstance(v, float):
        print(f"{k:24s} {v:+.4f}" if abs(v) < 1 else f"{k:24s} {v:.4f}")
    else:
        print(f"{k:24s} {v}")
# Error distributions (Dice/IoU/relations) -- the other three rubric criteria.
ed = SUMMARY.get("error_distributions", {})
ed_tot = ed.get("totals", {})
ed_iou = ed.get("iou", {})
ed_dice = ed.get("dice", {})
ed_rel = ed.get("relations", {})

METRICS.update({
    "ed_recall": ed_tot.get("recall"),
    "ed_precision": ed_tot.get("precision"),
    "iou_matched_mean": ed_iou.get("matched", {}).get("mean"),
    "iou_gt_mean": ed_iou.get("best_per_gt", {}).get("mean"),
    "iou_gt_frac_ge_075": ed_iou.get("best_per_gt", {}).get("frac_ge_0.75"),
    "iou_gt_frac_lt_010": ed_iou.get("best_per_gt", {}).get("frac_lt_0.10"),
    "dice_matched_mean": ed_dice.get("matched", {}).get("mean"),
    "dice_gt_mean": ed_dice.get("best_per_gt", {}).get("mean"),
    "dice_gt_frac_ge_075": ed_dice.get("best_per_gt", {}).get("frac_ge_0.75"),
    "dice_gt_frac_lt_010": ed_dice.get("best_per_gt", {}).get("frac_lt_0.10"),
    "o2m_multi_01": ed_rel.get("one_to_many_iou_0.1", {}).get("frac_multi"),
    "o2m_multi_05": ed_rel.get("one_to_many_iou_0.5", {}).get("frac_multi"),
    "m2o_multi_01": ed_rel.get("many_to_one_iou_0.1", {}).get("frac_multi"),
    "m2o_multi_05": ed_rel.get("many_to_one_iou_0.5", {}).get("frac_multi"),
})


n_phases                 4
n_accepted               1
n_rejected               2
best_pq                  +0.4386
best_phase               Candidate re-scoring
phase1_pq                +0.4220
phase2_pq                +0.4386
phase2_gain              +0.0166
phase2_paired            +0.0166
phase2_paired_ci         [0.0095, 0.0179]
phase3_delta             -0.0150
phase4_delta             -0.0113
public_lb                +0.3800
local_public_gap         +0.0586
oracle_pq                +0.5028
oracle_gap               +0.0808
captured_gap             +0.0166
remaining_gap            +0.0642
capture_fraction         +0.2054
n_candidates             6166
gbdt_corr                +0.5749
cnn_corr                 +0.5070
fragment_fraction        +0.0180
fragment_oracle_gain     +0.0015
pvt_corr                 +0.9132
pvt_solo                 +0.3906
conv_solo                +0.4177
train_files              707
test_files               180
n_annotations            8199
n_filaments_submitte

### Sanity checks

A report is only as good as the numbers behind it, so the derived values are
checked against the raw experiment files before anything is rendered. These
assertions are cheap and they catch the failure mode this project has hit
repeatedly: a number that looks plausible but came from a stale or leaky run.

In [7]:
checks = []

# Phase 2 must beat Phase 1, and the paired CI must exclude zero. The pooled
# CI is wide because it resamples images; the paired one is the honest test.
checks.append(("phase2 > phase1", p2["pq"] > p1["pq"]))
checks.append(("phase2 paired CI excludes 0",
               p2["paired_ci"][0] > 0 if p2.get("paired_ci") else False))

# Both rejected phases must actually be negative, or the report is lying
# about its own conclusions.
checks.append(("phase3 delta < 0", (p3.get("paired_delta") or 0) < 0))
checks.append(("phase4 delta < 0", (p4.get("paired_delta") or 0) < 0))

# The oracle ceiling must exceed the achieved score, otherwise the framing
# of "remaining headroom" is meaningless.
checks.append(("oracle > phase2",
               (detail.get("oracle_ceiling_pq") or 0) > p2["pq"]))

# Captured + remaining must reconstruct the oracle gap.
if detail.get("oracle_gap"):
    recon = detail["captured_gap"] + detail["remaining_gap"]
    checks.append(("captured + remaining == oracle gap",
                   abs(recon - detail["oracle_gap"]) < 1e-6))

# The fragment claim was wrong once (58.5% -> 1.8%). Guard it.
if fp.get("fragment_fraction") is not None:
    checks.append(("fragment fraction < 5%", fp["fragment_fraction"] < 0.05))

# The error distributions must reproduce the PQ totals exactly. If they do
# not, the post-processing used to compute them differs from the one that
# produced the reported PQ, and the figures describe a system we did not ship.
if ed_tot and fp:
    # fp_anatomy.json records the same TP count from an independent pass over
    # the same OOF dumps, so it is a genuine cross-check rather than a
    # restatement of the same computation.
    checks.append(("error dist TP == fp_anatomy TP",
                   ed_tot["n_tp"] == fp["tp"]))
    # fp_anatomy's three categories partition the FALSE POSITIVES, not the
    # false negatives: background + unmatched_gt + fragment == FP. Getting this
    # backwards is easy and the check caught it.
    checks.append(("fp_anatomy categories sum to FP",
                   fp["background"] + fp["unmatched_gt"] + fp["fragment"]
                   == ed_tot["n_pred"] - ed_tot["n_tp"]))
    checks.append(("error dist pred == TP + FP",
                   ed_tot["n_pred"]
                   == fp["tp"] + fp["background"] + fp["unmatched_gt"]
                   + fp["fragment"]))
    # Matched-pair mean IoU is SQ by definition.
    checks.append(("matched IoU mean == SQ",
                   abs(ed_iou["matched"]["mean"] - p1["sq"]) < 1e-4))
    # Dice = 2I/(1+I) is exact PER PAIR, but the mean of a nonlinear function
    # is not the function of the mean. Comparing the two means is therefore
    # wrong by ~0.003 (Jensen's inequality) and would fail a correct dataset.
    # The honest check is that Dice >= IoU everywhere, which follows from
    # Dice = 2I/(1+I) >= I for I in [0, 1].
    checks.append(("Dice >= IoU for every population",
                   all(ed_dice[v]["mean"] >= ed_iou[v]["mean"]
                       for v in ("matched", "best_per_gt", "best_per_pred"))))

for name, ok in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

assert all(ok for _, ok in checks), "sanity checks failed — do not publish"

PASS  phase2 > phase1
PASS  phase2 paired CI excludes 0
PASS  phase3 delta < 0
PASS  phase4 delta < 0
PASS  oracle > phase2
PASS  captured + remaining == oracle gap
PASS  fragment fraction < 5%
PASS  error dist TP == fp_anatomy TP
PASS  fp_anatomy categories sum to FP
PASS  error dist pred == TP + FP
PASS  matched IoU mean == SQ
PASS  Dice >= IoU for every population


## 6. Build the report draft template

The template is Jinja2. Two deliberate choices:

- **The results table is generated by a loop**, not written out row by row, so
  adding a phase to `summary.json` adds it to the report.
- **Negative results get their own section.** Phase 3 and Phase 4 both failed.
  Reporting them is more useful to a reader than quietly omitting them, and it
  is the honest thing to do when the rubric rewards a described pipeline over
  a lucky score.

In [8]:
TEMPLATE = """# {{ title }}

*{{ subtitle }}* — {{ author }}, {{ report_date }}

## Executive summary

We segment solar filaments in 2048x2048 H-alpha observations and score them with
Panoptic Quality. The final pipeline reaches **PQ {{ '%.4f'|format(m.phase2_pq) }}**
pooled out-of-fold over {{ m.train_files }} training files, a paired gain of
**{{ '%+.4f'|format(m.phase2_paired) }}** over the segmentation-only baseline
(95% CI {{ '[%+.4f, %+.4f]'|format(m.phase2_paired_ci[0], m.phase2_paired_ci[1]) }}).
The public leaderboard score is **{{ '%.2f'|format(m.public_lb) }}**.

The method has two stages. A five-fold ConvNeXt-Small UNet produces a
probability map; connected components of that map are then re-ranked by a
gradient-boosted model over 17 shape and intensity features. The second stage
is what earns the gain: it removes false positives without deleting true
filaments, which is exactly the trade PQ rewards.

Two further experiments were run and **both failed**. They are reported in full
below rather than omitted, because a negative result that is measured properly
is worth more than an unmeasured claim.

## Data overview

| Property | Value |
|---|---|
| Training files | {{ m.train_files }} |
| Training image entries (annotator copies) | {{ m.n_annotations and ds.train_image_entries or ds.train_image_entries }} |
| Annotations | {{ m.n_annotations }} |
| Test files | {{ m.test_files }} |
| Image size | {{ ds.image_size }}x{{ ds.image_size }}, {{ ds.image_mode }} |
| Median filament pixel fraction | {{ '%.2f'|format(100 * ds.filament_pixel_fraction_median) }}% |
| Median instance area | {{ ds.instance_area_px_median }} px |

Each image carries between one and three independent annotator copies, so the
same pixels are scored against several ground truths. This is why the metric is
pooled over copies rather than averaged per image: a prediction that satisfies
one annotator but not another should be penalised, and pooling does that.

## Results

| Phase | Change | PQ | SQ | RQ | Verdict |
|---|---|---|---|---|---|
{% for p in phases -%}
| {{ p.name }} | {{ p.change }} | {{ '%.4f'|format(p.pq) }} | {{ '%.4f'|format(p.sq) }} | {{ '%.4f'|format(p.rq) }} | {{ p.verdict }} |
{% endfor %}

SQ is segmentation quality (mean IoU of matched instances) and RQ is
recognition quality (how many instances were found at all). The split matters:
SQ barely moves across every architecture tried, which says the boundary
precision of the network is not the bottleneck. RQ is what the re-scoring stage
improves.

## Where the remaining error is

Scoring every candidate with its true label gives an oracle ceiling of
**PQ {{ '%.4f'|format(m.oracle_pq) }}**, i.e. **{{ '%+.4f'|format(m.oracle_gap) }}**
above the baseline. The learned scorer captures
**{{ '%+.4f'|format(m.captured_gap) }}** of that
({{ '%.0f'|format(100 * m.capture_fraction) }}%), leaving
**{{ '%+.4f'|format(m.remaining_gap) }}** on the table.

That gap is the single largest known opportunity in this project, and it is a
*ranking* problem rather than a segmentation problem: the candidates already
contain the right answer, the scorer just cannot tell which ones they are. The
GBDT reaches a correlation of {{ '%.3f'|format(m.gbdt_corr) }} with the true
candidate quality; a CNN over image crops did worse
({{ '%.3f'|format(m.cnn_corr) }}), which suggests the 17 scalar features already
encode most of what a crop would show.

## Negative results

### More data did not help

The competition images are GONG H-alpha frames, so annotations can be
transferred from nearby frames in the archive. This produced 3.1x the training
data and a **{{ '%+.4f'|format(m.phase3_delta) }}** change in PQ — a regression.

The per-year breakdown explains it. Gains appeared exactly where the extra
frames were concentrated (2021-22), and losses appeared in years where the
transferred labels came from a different observation of an evolving object.
Solar filaments change over hours; a polygon drawn on a frame six hours earlier
is not the same object. The model learned boundary imprecision.

### Architectural diversity did not help

Averaging a pyramid vision transformer (PVTv2-B2) with the ConvNeXt ensemble
gave **{{ '%+.4f'|format(m.phase4_delta) }}**. The mechanism is visible in one
number: the two models' predictions correlate at
**{{ '%.3f'|format(m.pvt_corr) }}**. They make the same mistakes, so averaging
them cannot cancel those mistakes — it just pulls the better model toward the
worse one (PVTv2 solo {{ '%.4f'|format(m.pvt_solo) }} against ConvNeXt
{{ '%.4f'|format(m.conv_solo) }}).

### A claim we had to retract

An early analysis asserted that fragments were 58.5% of false positives, which
would have justified a fragment-merging stage. Measured on 707 out-of-fold
files, the true figure is **{{ '%.1f'|format(100 * m.fragment_fraction) }}%**,
and the oracle gain from removing every fragment is
**{{ '%+.4f'|format(m.fragment_oracle_gain) }}**. The stage was cancelled before
it was built. The false positives are overwhelmingly background and unmatched
ground truth, not fragments.

## Error distributions

PQ is a single scalar and it hides the shape of the error. The rubric asks for
three further views, all measured here on the same 707 out-of-fold files with
the shipped post-processing.

### IoU and Dice

There are three populations and they answer different questions, so all three
are reported. Quoting only the first would flatter the system by hiding the
misses; quoting only the second would hide that boundaries are good when the
instance is found.

| Population | n | IoU mean | IoU p50 | Dice mean | Dice p50 | Dice >= 0.75 |
|---|---|---|---|---|---|---|
| Matched pairs (SQ view) | {{ '{:,}'.format(ed_iou.matched.n) }} | {{ '%.3f'|format(m.iou_matched_mean) }} | {{ '%.3f'|format(ed_iou.matched.p50) }} | {{ '%.3f'|format(m.dice_matched_mean) }} | {{ '%.3f'|format(ed_dice.matched.p50) }} | {{ '%.1f'|format(100 * ed_dice.matched['frac_ge_0.75']) }}% |
| Per ground-truth (misses = 0) | {{ '{:,}'.format(ed_iou.best_per_gt.n) }} | {{ '%.3f'|format(m.iou_gt_mean) }} | {{ '%.3f'|format(ed_iou.best_per_gt.p50) }} | {{ '%.3f'|format(m.dice_gt_mean) }} | {{ '%.3f'|format(ed_dice.best_per_gt.p50) }} | {{ '%.1f'|format(100 * m.dice_gt_frac_ge_075) }}% |
| Per prediction (FPs = 0) | {{ '{:,}'.format(ed_iou.best_per_pred.n) }} | {{ '%.3f'|format(ed_iou.best_per_pred.mean) }} | {{ '%.3f'|format(ed_iou.best_per_pred.p50) }} | {{ '%.3f'|format(ed_dice.best_per_pred.mean) }} | {{ '%.3f'|format(ed_dice.best_per_pred.p50) }} | {{ '%.1f'|format(100 * ed_dice.best_per_pred['frac_ge_0.75']) }}% |

The gap between the first two rows is the whole story. When an instance is
matched, Dice averages {{ '%.3f'|format(m.dice_matched_mean) }} and
{{ '%.1f'|format(100 * ed_dice.matched['frac_ge_0.75']) }}% of matches exceed 0.75 —
the boundaries are good. Across *all* ground-truth instances the mean falls to
{{ '%.3f'|format(m.dice_gt_mean) }}, because
{{ '%.1f'|format(100 * m.dice_gt_frac_lt_010) }}% of filaments score below 0.10,
i.e. were missed entirely.

So the error is dominated by **detection**, not delineation. That is consistent
with SQ being pinned near 0.68 across every architecture tried while RQ moves.

![IoU distribution](../figures/iou_distribution.png)

![Dice distribution](../figures/dice_distribution.png)

### One-to-many and many-to-one

PQ's denominator charges $0.5|FP| + 0.5|FN|$, so splitting one filament into
three costs almost as much as missing it. The relation histogram shows how often
that happens.

| Relation | IoU > 0.1 | IoU > 0.5 |
|---|---|---|
| One GT overlapped by k > 1 predictions (fragmentation) | {{ '%.1f'|format(100 * m.o2m_multi_01) }}% | {{ '%.1f'|format(100 * m.o2m_multi_05) }}% |
| One prediction overlapping k > 1 GT (merging) | {{ '%.1f'|format(100 * m.m2o_multi_01) }}% | {{ '%.1f'|format(100 * m.m2o_multi_05) }}% |

At the metric's own matching threshold of 0.5 there is **no** fragmentation or
merging at all: every overlap is a clean 1:1 or nothing. The multi-relations
that do exist live at loose overlaps (IoU > 0.1), where a prediction grazes a
neighbouring filament without competing for the match.

This is why the fragment-merging stage was cancelled. There is nothing to merge.

![Instance relations](../figures/instance_relations.png)

## Submission

{{ m.n_filaments_submitted }} filaments across {{ m.test_files }} test images,
with {{ m.n_empty_images }} images predicted empty. The local-to-public gap is
{{ '%.4f'|format(m.local_public_gap) }}, which is not a constant across
component types and should not be treated as a fixed offset.

## Reproducibility

Code, pinned requirements and a pipeline notebook are in the public repository.
Weights are published as release assets and fetched by
`scripts/fetch_weights.py`. Inference requires no network access: the
checkpoint supplies every weight, so the pretrained backbone is not downloaded.
"""

print(f"template: {len(TEMPLATE)} chars")

template: 8920 chars


## 7. Render the draft with data

Render and display inline. Reading the rendered Markdown before exporting is
the cheapest way to catch a template that silently produced `None` or a
mangled number.

In [9]:
env = jinja2.Environment(trim_blocks=True, lstrip_blocks=True)
rendered = env.from_string(TEMPLATE).render(
    title=CONFIG["title"],
    subtitle=CONFIG["subtitle"],
    author=CONFIG["author"],
    report_date=CONFIG["report_date"],
    m=METRICS,
    ds=SUMMARY["dataset"],
    phases=SUMMARY["phases"],
    ed_iou=SUMMARY["error_distributions"]["iou"],
    ed_dice=SUMMARY["error_distributions"]["dice"],
)

# A rendered template that leaked a Python repr is worse than no report.
for bad in ("None", "nan", "{{ ", "{% "):
    assert bad not in rendered, f"template leaked {bad!r}"

print(f"rendered {len(rendered)} chars, {rendered.count(chr(10))} lines")
display(Markdown(rendered))

rendered 7396 chars, 172 lines


# Solar Filament Segmentation with Panoptic Quality

*MAGFiLO 2026 Challenge — Technical Report* — Yasir Latif, 2026-11-09

## Executive summary

We segment solar filaments in 2048x2048 H-alpha observations and score them with
Panoptic Quality. The final pipeline reaches **PQ 0.4386**
pooled out-of-fold over 707 training files, a paired gain of
**+0.0166** over the segmentation-only baseline
(95% CI [+0.0095, +0.0179]).
The public leaderboard score is **0.38**.

The method has two stages. A five-fold ConvNeXt-Small UNet produces a
probability map; connected components of that map are then re-ranked by a
gradient-boosted model over 17 shape and intensity features. The second stage
is what earns the gain: it removes false positives without deleting true
filaments, which is exactly the trade PQ rewards.

Two further experiments were run and **both failed**. They are reported in full
below rather than omitted, because a negative result that is measured properly
is worth more than an unmeasured claim.

## Data overview

| Property | Value |
|---|---|
| Training files | 707 |
| Training image entries (annotator copies) | 1154 |
| Annotations | 8199 |
| Test files | 180 |
| Image size | 2048x2048, 8-bit grayscale |
| Median filament pixel fraction | 0.31% |
| Median instance area | 1228 px |

Each image carries between one and three independent annotator copies, so the
same pixels are scored against several ground truths. This is why the metric is
pooled over copies rather than averaged per image: a prediction that satisfies
one annotator but not another should be penalised, and pooling does that.

## Results

| Phase | Change | PQ | SQ | RQ | Verdict |
|---|---|---|---|---|---|
| Segmentation baseline | 5-fold ConvNeXt-Small UNet, 512px patches, disk-flattened input | 0.4220 | 0.6765 | 0.6238 | baseline |
| Candidate re-scoring | Re-rank connected components with a GBDT over 17 shape/intensity features | 0.4386 | 0.6786 | 0.6464 | accepted |
| GONG label transfer | Transfer annotations from nearby GONG frames, 3.1x training data | 0.4045 | 0.6667 | 0.6095 | rejected |
| PVTv2-B2 ensemble diversity | Add five PVTv2-B2 folds to the ConvNeXt ensemble | 0.4098 | 0.6719 | 0.6099 | rejected |

SQ is segmentation quality (mean IoU of matched instances) and RQ is
recognition quality (how many instances were found at all). The split matters:
SQ barely moves across every architecture tried, which says the boundary
precision of the network is not the bottleneck. RQ is what the re-scoring stage
improves.

## Where the remaining error is

Scoring every candidate with its true label gives an oracle ceiling of
**PQ 0.5028**, i.e. **+0.0808**
above the baseline. The learned scorer captures
**+0.0166** of that
(21%), leaving
**+0.0642** on the table.

That gap is the single largest known opportunity in this project, and it is a
*ranking* problem rather than a segmentation problem: the candidates already
contain the right answer, the scorer just cannot tell which ones they are. The
GBDT reaches a correlation of 0.575 with the true
candidate quality; a CNN over image crops did worse
(0.507), which suggests the 17 scalar features already
encode most of what a crop would show.

## Negative results

### More data did not help

The competition images are GONG H-alpha frames, so annotations can be
transferred from nearby frames in the archive. This produced 3.1x the training
data and a **-0.0150** change in PQ — a regression.

The per-year breakdown explains it. Gains appeared exactly where the extra
frames were concentrated (2021-22), and losses appeared in years where the
transferred labels came from a different observation of an evolving object.
Solar filaments change over hours; a polygon drawn on a frame six hours earlier
is not the same object. The model learned boundary imprecision.

### Architectural diversity did not help

Averaging a pyramid vision transformer (PVTv2-B2) with the ConvNeXt ensemble
gave **-0.0113**. The mechanism is visible in one
number: the two models' predictions correlate at
**0.913**. They make the same mistakes, so averaging
them cannot cancel those mistakes — it just pulls the better model toward the
worse one (PVTv2 solo 0.3906 against ConvNeXt
0.4177).

### A claim we had to retract

An early analysis asserted that fragments were 58.5% of false positives, which
would have justified a fragment-merging stage. Measured on 707 out-of-fold
files, the true figure is **1.8%**,
and the oracle gain from removing every fragment is
**+0.0015**. The stage was cancelled before
it was built. The false positives are overwhelmingly background and unmatched
ground truth, not fragments.

## Error distributions

PQ is a single scalar and it hides the shape of the error. The rubric asks for
three further views, all measured here on the same 707 out-of-fold files with
the shipped post-processing.

### IoU and Dice

There are three populations and they answer different questions, so all three
are reported. Quoting only the first would flatter the system by hiding the
misses; quoting only the second would hide that boundaries are good when the
instance is found.

| Population | n | IoU mean | IoU p50 | Dice mean | Dice p50 | Dice >= 0.75 |
|---|---|---|---|---|---|---|
| Matched pairs (SQ view) | 5,259 | 0.676 | 0.682 | 0.804 | 0.811 | 77.5% |
| Per ground-truth (misses = 0) | 8,199 | 0.501 | 0.598 | 0.614 | 0.749 | 49.7% |
| Per prediction (FPs = 0) | 8,662 | 0.481 | 0.583 | 0.591 | 0.737 | 47.0% |

The gap between the first two rows is the whole story. When an instance is
matched, Dice averages 0.804 and
77.5% of matches exceed 0.75 —
the boundaries are good. Across *all* ground-truth instances the mean falls to
0.614, because
16.7% of filaments score below 0.10,
i.e. were missed entirely.

So the error is dominated by **detection**, not delineation. That is consistent
with SQ being pinned near 0.68 across every architecture tried while RQ moves.

![IoU distribution](../figures/iou_distribution.png)

![Dice distribution](../figures/dice_distribution.png)

### One-to-many and many-to-one

PQ's denominator charges $0.5|FP| + 0.5|FN|$, so splitting one filament into
three costs almost as much as missing it. The relation histogram shows how often
that happens.

| Relation | IoU > 0.1 | IoU > 0.5 |
|---|---|---|
| One GT overlapped by k > 1 predictions (fragmentation) | 4.8% | 0.0% |
| One prediction overlapping k > 1 GT (merging) | 1.8% | 0.0% |

At the metric's own matching threshold of 0.5 there is **no** fragmentation or
merging at all: every overlap is a clean 1:1 or nothing. The multi-relations
that do exist live at loose overlaps (IoU > 0.1), where a prediction grazes a
neighbouring filament without competing for the match.

This is why the fragment-merging stage was cancelled. There is nothing to merge.

![Instance relations](../figures/instance_relations.png)

## Submission

1093 filaments across 180 test images,
with 3 images predicted empty. The local-to-public gap is
0.0586, which is not a constant across
component types and should not be treated as a fixed offset.

## Reproducibility

Code, pinned requirements and a pipeline notebook are in the public repository.
Weights are published as release assets and fetched by
`scripts/fetch_weights.py`. Inference requires no network access: the
checkpoint supplies every weight, so the pretrained backbone is not downloaded.

## 8. Export the draft to Markdown and HTML

Both formats are written. Markdown is the editable source; HTML is what a
reviewer can open without a Markdown viewer.

The HTML gets a minimal stylesheet because the default `markdown` output is
unstyled and the tables are unreadable without one.

In [10]:
CONFIG["out_md"].write_text(rendered, encoding="utf-8")

body = markdown.markdown(rendered, extensions=["tables", "fenced_code"])
html = f"""<!doctype html>
<html><head><meta charset="utf-8">
<title>{CONFIG['title']}</title>
<style>
  body {{ max-width: 52rem; margin: 3rem auto; padding: 0 1.5rem;
         font: 16px/1.6 -apple-system, Segoe UI, Roboto, sans-serif;
         color: #1a1a1a; }}
  h1 {{ border-bottom: 2px solid #ddd; padding-bottom: .4rem; }}
  h2 {{ margin-top: 2.2rem; border-bottom: 1px solid #eee; }}
  table {{ border-collapse: collapse; margin: 1rem 0; font-size: 14px; }}
  th, td {{ border: 1px solid #ddd; padding: .4rem .7rem; text-align: left; }}
  th {{ background: #f6f6f6; }}
  code {{ background: #f3f3f3; padding: .1rem .3rem; border-radius: 3px; }}
  em {{ color: #555; }}
</style></head><body>
{body}
</body></html>"""
CONFIG["out_html"].write_text(html, encoding="utf-8")

for key in ("out_md", "out_html"):
    p = CONFIG[key]
    print(f"{p.resolve()}  ({p.stat().st_size:,} bytes)")

/media/yasir/HDD/drive/repos/competitions/solar/v2/results/report/report_draft.md  (7,406 bytes)
/media/yasir/HDD/drive/repos/competitions/solar/v2/results/report/report_draft.html  (9,511 bytes)


## Next steps

This draft covers the quantitative results and the pipeline. Before submission
it still needs:

1. **A pipeline figure** — preprocessing, network, candidate re-scoring, RLE.
2. **Dice and IoU distributions** — explicitly scored in the quantitative 70%,
   and not yet computed.
3. **A one-to-many / many-to-one histogram** — also explicitly scored, also not
   yet computed.
4. **Formatting to the Overleaf template** — the competition requires 4 pages
   in their LaTeX template, not Markdown.

Items 2 and 3 are the highest value: they are named in the rubric and we have
the out-of-fold predictions needed to compute them.